In [1]:
# Imports
import tensorflow as tf

from tensorflow.keras import models
from tensorflow.keras import layers

tf.random.set_seed(666)

In [2]:
# Load the FashionMNIST dataset, scale the pixel values
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
X_train = X_train/255.
X_test = X_test/255.

X_train.shape, X_test.shape, y_train.shape, y_test.shape

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


((60000, 28, 28), (10000, 28, 28), (60000,), (10000,))

In [3]:
# Change the pixel values to float32 and reshape input data
X_train = X_train.astype("float32").reshape(-1, 28, 28, 1)
X_test = X_test.astype("float32").reshape(-1, 28, 28, 1)

In [4]:
# Define utility function for building a basic shallow Convnet
def get_teacher_model():
    model = models.Sequential()
    model.add(layers.Conv2D(16, (5, 5), activation="relu",
        input_shape=(28, 28, 1)))
    model.add(layers.MaxPooling2D(pool_size=(2, 2)))
    model.add(layers.Conv2D(32, (5, 5), activation="relu"))
    model.add(layers.MaxPooling2D(pool_size=(2, 2)))
    model.add(layers.Dropout(0.2))
    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation="relu"))
    model.add(layers.Dense(10))

    return model

In [5]:
# Define loss function and optimizer
loss_func = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam()

In [6]:
# Prepare TF dataset
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(100).batch(64)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(64)

# Train the teacher model
teacher_model = get_teacher_model()
teacher_model.compile(loss=loss_func, optimizer=optimizer, metrics=["accuracy"])
teacher_model.fit(train_ds,
                  validation_data=test_ds,
                  epochs=10)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.7188 - loss: 0.7830 - val_accuracy: 0.8457 - val_loss: 0.4313
Epoch 2/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8547 - loss: 0.4005 - val_accuracy: 0.8667 - val_loss: 0.3621
Epoch 3/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8759 - loss: 0.3410 - val_accuracy: 0.8770 - val_loss: 0.3359
Epoch 4/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8881 - loss: 0.3100 - val_accuracy: 0.8880 - val_loss: 0.3084
Epoch 5/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8968 - loss: 0.2809 - val_accuracy: 0.8872 - val_loss: 0.3097
Epoch 6/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9011 - loss: 0.2656 - val_accuracy: 0.8944 - val_loss: 0.2909
Epoch 7/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9076 - loss: 0.2521 - val_accuracy: 0.8977 - val_loss: 0.2772
Epoch 8/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9115 - loss: 0.2391 - val_accuracy: 0

In [8]:
# Evaluate and serialize
print("Test accuracy: {:.2f}".format(teacher_model.evaluate(test_ds)[1]*100))
teacher_model.save_weights("teacher_model.weights.h5")

157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9032 - loss: 0.2707
Test accuracy: 90.40


In [9]:
# Student model utility
def get_student_model():
    model = models.Sequential()
    model.add(layers.Input(shape=(28, 28, 1)))
    model.add(layers.Flatten())
    model.add(layers.Dense(48, activation="relu"))
    model.add(layers.Dense(10))

    return model

In [10]:
# Credits: https://github.com/google-research/simclr/blob/master/colabs/distillation_self_training.ipynb
def get_kd_loss(student_logits, teacher_logits, temperature=0.5):
    teacher_probs = tf.nn.softmax(teacher_logits / temperature)
    kd_loss = tf.compat.v1.losses.softmax_cross_entropy(
        teacher_probs, student_logits / temperature, temperature**2)
    return kd_loss

In [11]:
# Model, optimizer
student_model = get_student_model()
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)

# Average the loss across the batch size within an epoch
train_loss = tf.keras.metrics.Mean(name="train_loss")
valid_loss = tf.keras.metrics.Mean(name="test_loss")

# Specify the performance metric
train_acc = tf.keras.metrics.SparseCategoricalAccuracy(name="train_acc")
valid_acc = tf.keras.metrics.SparseCategoricalAccuracy(name="valid_acc")

In [29]:
# Train utils
@tf.function
def model_train(images, labels, teacher_model,
                student_model, optimizer, temperature):
    teacher_logits = teacher_model(images)

    with tf.GradientTape() as tape:
        student_logits = student_model(images)
        loss = get_kd_loss(student_logits, teacher_logits, temperature)

    gradients = tape.gradient(loss, student_model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, student_model.trainable_variables))

    train_loss(loss)
    train_acc(labels, tf.nn.softmax(student_logits))

In [30]:
# Validation utils
@tf.function
def model_validate(images, labels, teacher_model,
                   student_model, temperature):
    teacher_logits = teacher_model(images)

    student_logits = student_model(images)
    loss = get_kd_loss(student_logits, teacher_logits, temperature)

    valid_loss(loss)
    valid_acc(labels, tf.nn.softmax(student_logits))

In [31]:
# Tie everything together
def train_model(epochs, teacher_model, student_model, optimizer, temperature=0.5):
    global train_loss, train_acc, valid_loss, valid_acc

    for epoch in range(epochs):
        # Re-initialize metrics to reset their states for the new epoch
        train_loss = tf.keras.metrics.Mean(name="train_loss")
        train_acc = tf.keras.metrics.SparseCategoricalAccuracy(name="train_acc")
        valid_loss = tf.keras.metrics.Mean(name="test_loss")
        valid_acc = tf.keras.metrics.SparseCategoricalAccuracy(name="valid_acc")

        for (images, labels) in train_ds:
            model_train(images, labels, teacher_model, student_model, optimizer, temperature)

        for (images, labels) in test_ds:
            model_validate(images, labels, teacher_model, student_model, temperature)

        (loss, acc) = train_loss.result(), train_acc.result()
        (val_loss, val_acc) = valid_loss.result(), valid_acc.result()

        template = "Epoch {}, loss: {:.3f}, acc: {:.3f}, val_loss: {:.3f}, val_acc: {:.3f}"
        print (template.format(epoch+1,
                            loss,
                            acc,
                            val_loss,
                            val_acc))


    return teacher_model, student_model

In [20]:
_, student_model = train_model(10, teacher_model, student_model, optimizer)

Epoch 1, loss: 0.000, acc: 0.000, val_loss: 0.000, val_acc: 0.000
Epoch 2, loss: 0.000, acc: 0.000, val_loss: 0.000, val_acc: 0.000
Epoch 3, loss: 0.000, acc: 0.000, val_loss: 0.000, val_acc: 0.000
Epoch 4, loss: 0.000, acc: 0.000, val_loss: 0.000, val_acc: 0.000
Epoch 5, loss: 0.000, acc: 0.000, val_loss: 0.000, val_acc: 0.000
Epoch 6, loss: 0.000, acc: 0.000, val_loss: 0.000, val_acc: 0.000
Epoch 7, loss: 0.000, acc: 0.000, val_loss: 0.000, val_acc: 0.000
Epoch 8, loss: 0.000, acc: 0.000, val_loss: 0.000, val_acc: 0.000
Epoch 9, loss: 0.000, acc: 0.000, val_loss: 0.000, val_acc: 0.000
Epoch 10, loss: 0.000, acc: 0.000, val_loss: 0.000, val_acc: 0.000


This can be further improved with longer training time and more careful hyperparameter tuning.

In [22]:
# Serialize
student_model.save_weights("student_model.weights.h5")

In [23]:
# Investigate the sizes
!ls -lh *.h5

-rw-r--r-- 1 root root 166K Dec 19 11:37 student_model.weights.h5
-rw-r--r-- 1 root root 975K Dec 19 11:33 teacher_model.weights.h5


Let's check the total number of trainable params.

In [24]:
teacher_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 24, 24, 16)     │           416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 12, 12, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 8, 8, 32)       │        12,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 4, 4, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 4, 4, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 240,608 (939.88 KB)

 Trainable params: 80,202 (313.29 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 160,406 (626.59 KB)

In [25]:
student_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 48)             │        37,680 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │           490 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 38,170 (149.10 KB)

 Trainable params: 38,170 (149.10 KB)

 Non-trainable params: 0 (0.00 B)

Further size decrease is possible with TFLite.

In [26]:
# Credits: https://www.tensorflow.org/lite/performance/post_training_quant

def representative_data_gen():
    for input_value in tf.data.Dataset.from_tensor_slices(X_train).batch(1).take(100):
        yield [input_value]

def convert_to_tflite(model, tflite_file):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_data_gen
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    tflite_quant_model = converter.convert()

    open(tflite_file, 'wb').write(tflite_quant_model)

In [27]:
convert_to_tflite(teacher_model, "teacher.tflite")
convert_to_tflite(student_model, "student.tflite")

Saved artifact at '/tmp/tmpgbzv3hbx'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  138022594562960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138019987281424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138019987281040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138019987281616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138019987285072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138019987285840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138019987286800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138019987284880: TensorSpec(shape=(), dtype=tf.resource, name=None)


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Saved artifact at '/tmp/tmpa3c9k5vd'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='keras_tensor_37')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  138019897643408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138019897645904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138019897644560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138019897644944: TensorSpec(shape=(), dtype=tf.resource, name=None)


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [28]:
!ls -lh *.tflite

-rw-r--r-- 1 root root 41K Dec 19 11:37 student.tflite
-rw-r--r-- 1 root root 87K Dec 19 11:37 teacher.tflite
